
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo - Chunking PDFs and AI Search
## Overview
In this demo, we'll take a PDF through the **Parse → Prep Search → AI Search** pipeline. We'll use `ai_parse_document` to extract text, `ai_prep_search` to chunk it, and create a AI Search index so the data is retrievable by a Knowledge Assistant.

## Learning Objectives
By the end of this demo, you will be able to:
1. Parse a PDF and chunk the text using `ai_prep_search`
2. Write chunked data to a Delta table with Change Data Feed enabled
3. Create a AI Search index on the `vs_all_demo` endpoint
4. Test the index with a similarity search query

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Beta Features</strong>
<p style="margin: 8px 0 0 0; color: #333;">This notebook uses Databricks Beta Features. While this notebook has been thoroughly tested, it's worth noting that full functionality is not guaranteed.</p>
</div>
</div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup

In [0]:
%run ./Includes/Classroom-Setup-06

As a part of the workspace setup, certain assets have been managed for you (volumes, PDF, etc.). Here, we read in from the shared catalog `dbacademy`, schema `ka_all`, and volume `vol_all` the PDF `airbnb_house_rules.pdf`.

In [0]:
HOUSE_RULES_PDF = "/Volumes/dbacademy/ka_all/vol_all/additional_listing_info/airbnb_house_rules.pdf"
CHUNKS_TABLE = f"{catalog_name}.{schema_name}.house_rules_chunks"

EMBEDDING_MODEL = "databricks-gte-large-en"
VS_ENDPOINT = "vs_all_demo"
VS_INDEX = f"{catalog_name}.{schema_name}.house_rules_index"

print(f"Source PDF:    {HOUSE_RULES_PDF}")
print(f"Chunks table:  {CHUNKS_TABLE}")
print(f"VS Index:      {VS_INDEX}")

## B. Parse the PDF

First, let's parse the house rules PDF using `ai_parse_document`. Second, view the stored table by reading it in using a **SELECT** SQL query.

In [0]:
parsed_df = spark.sql(f"""
  WITH parsed AS (
    SELECT
      path,
      ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM READ_FILES('{HOUSE_RULES_PDF}', format => 'binaryFile')
  )
  SELECT
    path,
    element.value:type::STRING AS element_type,
    element.value:content::STRING AS content
  FROM parsed,
    LATERAL variant_explode(parsed.parsed:document:elements) AS element
  WHERE element.value:content IS NOT NULL
""")

print(f"Parsed {parsed_df.count()} elements from the PDF")
parsed_df.write.mode("overwrite").saveAsTable(f"parsed_house_rules")

In [0]:
%sql
SELECT * FROM parsed_house_rules

## C. Chunk with `ai_prep_search`

Now we'll use `ai_prep_search` to intelligently chunk the parsed text. This function uses AI models to split text at semantically meaningful boundaries, which is better than simple fixed-size splitting.

`ai_prep_search()` needs the full `ai_parse_document` `VARIANT` per document, not the flattened elements shown above in `parsed_house_rules`. This means you need to run it on the original parsed output.

1. Run the next cell to save the results from using `ai_prep_search()` to `house_rules_chunks`.

In [0]:
chunks_df = spark.sql(f"""
  WITH parsed AS (
    SELECT
      path,
      ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM READ_FILES('{HOUSE_RULES_PDF}', format => 'binaryFile')
  ),
  prepped AS (
    SELECT
      path,
      ai_prep_search(parsed) AS prepped
    FROM parsed
  )
  SELECT
    p.path,
    chunk.value:chunk_id::STRING        AS chunk_id,
    chunk.value:chunk_position::INT     AS chunk_position,
    chunk.value:chunk_to_retrieve::STRING AS chunk_to_retrieve,
    chunk.value:chunk_to_embed::STRING  AS chunk_to_embed
  FROM prepped p,
       LATERAL variant_explode(p.prepped:document:contents) AS chunk
""")

chunks_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("house_rules_chunks")

### C1. Understanding the Output
The output from running the next cell will have two columns of particular interest as we look to build our AI Search Index:
- `chunk_to_retrieve`: This column is what you show the LLM after retrieval. It's the raw (or near-raw) chunk text, preserving full content and structure so the model has everything it needs to answer questions. 
- `chunk_to_embed`: This column is what you send to the embedding model. It's a cleaned, context-enriched string that prepends document-level metadata (title, section headers, page number, captions, etc.) to the chunk text, optimized purely for retrieval quality.

2. In the first row of the produced table, click the dropdown to expand the `chunk_to_retrieve` and `chunk_to_embed` columns. Inspect the results and recognize that the raw text is stored in `chunk_to_retrieved` while `chunk_to_embed` is the context-enriched text prepared for embeddings. To see the format of `chunk_to_embed`, please see this documentation ([AWS](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_prep_search#chunk_to_embed-format) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/ai_prep_search#chunk_to_embed-format) | [GCP](https://docs.databricks.com/gcp/en/sql/language-manual/functions/ai_prep_search#chunk_to_embed-format)).

In [0]:
%sql
SELECT * FROM house_rules_chunks

## D. Write Chunks to Unity Catalog

We need to persist the chunks in a Delta table with Change Data Feed (CDF) enabled (this is required for AI Search to sync automatically). CDF tracks and records all changes (inserts, updates, deletes) to the table, allowing downstream systems to efficiently detect and process data modifications.

In the next cell, this is accomplished by writing the chunks DataFrame to a Delta table with the `delta.enableChangeDataFeed` property set to `true`. This enables CDF, allowing AI Search to sync new and updated chunks automatically.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">Learn more about CDF</strong>
<p style="margin: 8px 0 0 0; color: #333;">
See the Databricks documentation on Change Data Feed (<a href="https://docs.databricks.com/aws/en/delta/delta-change-data-feed.html" style="color: #1976d2;">AWS</a> | <a href="https://docs.databricks.com/gcp/en/delta/delta-change-data-feed.html" style="color: #1976d2;">GCP</a>) for a deeper dive.
</p>
</div>

In [0]:
from pyspark.sql.functions import expr, lit, col

# Map ai_prep_search outputs into your chunks table schema
chunks_with_id = (
    chunks_df
    .withColumn("chunk_id", expr("uuid()"))            # PK for the table / index
    .withColumn("source_path", col("path"))
    .select(
        "chunk_id",
        col("chunk_to_embed").alias("chunk_text"),     # for embeddings
        col("chunk_to_retrieve").alias("retrieval_text"),
        "source_path"
    )
)

chunks_with_id.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(CHUNKS_TABLE)

spark.sql(f"ALTER TABLE {CHUNKS_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print(f"Chunks table written: {CHUNKS_TABLE}")
display(spark.table(CHUNKS_TABLE))

## E. Create AI Search Index

Now we'll attach a AI Search index to the `vs_all_demo` endpoint. This index will use managed embeddings: Databricks automatically computes embeddings from the `chunk_text` column.

**Note:** As part of the workspace setup, the `vs_all_demo` endpoint should already be ready by the time you reach this point of this notebook. You can validate that the endpoint exists and is ready by navigating to **Compute > AI Search** and view the **Status** displayed there.

In [0]:
from databricks.vector_search.client import VectorSearchClient
import time

vsc = VectorSearchClient()

# Verify the endpoint is ready
for _ in range(150):
    try:
        ep = vsc.get_endpoint(VS_ENDPOINT)
        status = ep.get("endpoint_status", {}).get("state", "")
        if status == "ONLINE":
            print(f"Endpoint '{VS_ENDPOINT}' is ONLINE.")
            break
        print(f"  Endpoint status: {status} - waiting...")
        time.sleep(10)
    except Exception:
        print(f"  Endpoint '{VS_ENDPOINT}' not found yet - waiting...")
        time.sleep(10)
else:
    print("WARNING: Endpoint not ready. The index creation may fail.")

In [0]:
# Create the index if it doesn't already exist
try:
    index = vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        source_table_name=CHUNKS_TABLE,
        index_name=VS_INDEX,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="chunk_text",
        embedding_model_endpoint_name=EMBEDDING_MODEL,
        columns_to_sync=["chunk_id", "chunk_text", "source_path"],
    )
    print(f"Index '{VS_INDEX}' created and syncing.")
except Exception as e:
    if "already exists" in str(e):
        index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX)
        print(f"Index '{VS_INDEX}' already exists. Syncing latest data.")
        index.sync()
    else:
        raise

print(index.describe())


## F. Test the Index

Once the index is online, let's run a quick similarity search to verify it works. The next cell does the following:

1. **Retrieves the index** from the AI Search client using the endpoint and index name defined earlier.
2. **Checks the index status** by calling `index.describe()` and printing the `detailed_state` — this tells you whether the index is still syncing or ready to serve queries.
3. **Runs a similarity search** using `index.similarity_search()` with the natural language query *"What are the house rules about pets?"* — the index converts this to an embedding and finds the 3 most semantically similar chunks.
4. **Prints the matching chunks** so you can verify the retrieved text is relevant to the query.

If the index is still syncing, re-run the cell after a few minutes.

In [0]:
index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX)
status = index.describe().get("status", {}).get("detailed_state")
print(f"Index status: {status}")

results = index.similarity_search(
    query_text="What are the house rules about pets?",
    columns=["chunk_id", "chunk_text"],
    num_results=3,
)
for row in results.get("result", {}).get("data_array", []):
    print(f"\n--- Chunk ---")
    print(row[1])

## G. Conclusion
In this demo, you:

1. **Parsed** a PDF with `ai_parse_document`
2. **Chunked** the text with `ai_prep_search` for semantically meaningful splits
3. **Stored** chunks in a Delta table with Change Data Feed enabled
4. **Created** a AI Search index on the shared `vs_all_demo` endpoint
5. **Tested** the index with a similarity search query

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
